# 01 — Noise schedule & forward corruption

Visualize the cosine and linear noise schedules, forward corruption of a toy latent, and the v-prediction target.

ALD-SC uses standard isotropic noise on the spatial latent $z$ (no metric-matched corruption). The spectral chart $c_\mathrm{spec}$ enters as conditioning, not as the corruption metric.

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import matplotlib.pyplot as plt
from ald_sc.schedule import CosineSchedule, LinearSchedule

torch.manual_seed(3407)

## 1. Schedule comparison: $\bar\alpha_t$

In [ ]:
cos = CosineSchedule(num_steps=1000)
lin = LinearSchedule(num_steps=1000)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cos.alpha_bar.numpy(), label="Cosine", linewidth=2)
ax.plot(lin.alpha_bar.numpy(), label="Linear (DDPM)", linewidth=2)
ax.set_xlabel("Timestep t")
ax.set_ylabel(r"$\bar{\alpha}_t$")
ax.set_title("Noise schedules")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../results/01_alpha_bar.png", dpi=150)
plt.show()

## 2. Forward corruption of a toy latent

Take a small 4-channel 16×16 latent and corrupt it at increasing timesteps.

In [ ]:
sched = CosineSchedule(num_steps=1000)
z0 = torch.randn(1, 4, 16, 16)
noise = torch.randn_like(z0)

timesteps = [0, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(timesteps), figsize=(18, 3))
for i, t in enumerate(timesteps):
    t_tensor = torch.tensor([t])
    z_t = sched.add_noise(z0, t_tensor, noise)
    # Show channel 0
    axes[i].imshow(z_t[0, 0].numpy(), cmap="viridis")
    axes[i].set_title(f"t={t}")
    axes[i].axis("off")
plt.suptitle("Forward corruption (cosine schedule)")
plt.tight_layout()
plt.savefig("../results/01_corruption_grid.png", dpi=150)
plt.show()

## 3. v-prediction target vs noise

The v-target is $v = \sqrt{\bar\alpha_t}\,\epsilon - \sqrt{1-\bar\alpha_t}\,z_0$.

The round-trip recovers $z_0$:
$z_0 = \sqrt{\bar\alpha_t}\,z_t - \sqrt{1-\bar\alpha_t}\,v$

In [ ]:
z0 = torch.randn(1, 4, 16, 16)
noise = torch.randn_like(z0)
t = torch.tensor([500])

z_t = sched.add_noise(z0, t, noise)
v = sched.v_target(z0, t, noise)

ab = sched.alpha_bar[t]
sqrt_ab = ab.sqrt().view(-1, 1, 1, 1)
sqrt_1mab = (1 - ab).sqrt().view(-1, 1, 1, 1)
z0_recovered = sqrt_ab * z_t - sqrt_1mab * v

print(f"alpha_bar(t=500) = {ab[0]:.4f}")
print(f"Round-trip error: {(z0_recovered - z0).abs().max():.2e}")
print(f"v has zero mean: {v.mean():.4f}")

## 4. DDIM-style sigma subsampling

Deterministic samplers use a subsampled set of timesteps.

In [ ]:
sigmas = sched.sample_sigmas(steps=20)
print(f"20-step sigma indices: {sigmas.tolist()}")
print(f"First: t={sigmas[0]}, Last: t={sigmas[-1]}")